# Análise Financeira - ClearBank

Notebook para ler, validar e analisar as transações bancárias do arquivo CSV.


## Criando o arquivo de transações

In [1]:
# criando o csv com dados de teste
# 16 registros validos em 5 meses, 5 invalidos e 2 suspeitos acima de R$ 10.000

dados = """id,data,cliente_id,tipo,valor,descricao,categoria
1,2026-01-05,CLI001,credito,3500.00,Salario janeiro,salario
2,2026-01-12,CLI002,debito,180.50,Supermercado,compra
3,2026-01-20,CLI001,debito,abc,Erro no sistema,compra
4,2026-02-03,,debito,200.00,Sem cliente,transferencia
5,2026-02-14,CLI003,credito,15000.00,Transferencia grande,transferencia
6,2026-02-18,CLI002,debito,320.00,Conta de luz,conta
7,2026-03-01,CLI001,credito,3500.00,Salario marco,salario
8,2026-03-10,CLI003,debito,99.90,Streaming,assinatura
9,2026-03-15,CLI002,credito,2800.00,Freelance,salario
10,2026-03-22,CLI001,debito,450.00,Aluguel,compra
11,2026-04-01,CLI001,credito,3500.00,Salario abril,salario
12,2026-04-08,CLI002,debito,75.00,Farmacia,compra
13,2026-04-14,CLI003,credito,12500.00,Resgate investimento,transferencia
14,2026-04-18,CLI001,debito,890.00,Passagem,compra
15,2026-04-25,CLI002,credito,1200.00,Bonus,salario
16,2026-05-02,CLI003,debito,340.00,Restaurante,compra
17,2026-05-10,CLI001,credito,3500.00,Salario maio,salario
18,2026-05-15,CLI002,debito,60.00,Streaming,assinatura
19,,CLI001,debito,100.00,Data invalida,compra
20,2026-05-20,CLI003,tipo_errado,200.00,Tipo invalido,compra
21,2026-05-22,CLI002,debito,-50.00,Valor negativo,compra"""

with open('transacoes.csv', 'w', encoding='utf-8') as f:
    f.write(dados)

print("arquivo criado!")


arquivo criado!


## Importações

In [2]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00


## Funções

In [3]:
def brl(valor):
    # formata valor no padrao monetario brasileiro
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def ler_transacoes(arquivo):
    transacoes = []
    try:
        with open(arquivo, newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for linha in reader:
                transacoes.append(dict(linha))
    except FileNotFoundError:
        print(f"arquivo {arquivo} nao encontrado")
    return transacoes


def validar_transacao(linha):
    # valida cada campo e retorna None se algo estiver errado

    # id precisa ser numero
    try:
        int(linha['id'])
    except (ValueError, KeyError):
        return None

    # cliente nao pode ser vazio
    if not linha.get('cliente_id', '').strip():
        return None

    # data precisa estar no formato certo
    try:
        data = datetime.strptime(linha['data'].strip(), '%Y-%m-%d')
    except (ValueError, KeyError, AttributeError):
        return None

    # tipo so pode ser credito ou debito
    tipo = linha.get('tipo', '').strip().lower()
    if tipo not in ['credito', 'debito']:
        return None

    # valor precisa ser numero maior que zero
    try:
        valor = float(linha['valor'])
        if valor <= 0:
            return None
    except (ValueError, KeyError):
        return None

    return {
        'id': int(linha['id']),
        'data': data,
        'cliente_id': linha['cliente_id'].strip(),
        'tipo': tipo,
        'valor': valor,
        'descricao': linha.get('descricao', ''),
        'categoria': linha.get('categoria', ''),
        'mes': data.strftime('%Y-%m')
    }


def gerar_relatorio(validas):
    resumo = {}

    for t in validas:
        mes = t['mes']

        if mes not in resumo:
            resumo[mes] = {
                'quantidade': 0,
                'total_credito': 0,
                'total_debito': 0,
                'valores': []
            }

        resumo[mes]['quantidade'] += 1
        resumo[mes]['valores'].append(t['valor'])

        if t['tipo'] == 'credito':
            resumo[mes]['total_credito'] += t['valor']
        else:
            resumo[mes]['total_debito'] += t['valor']

    # calcula as metricas finais de cada mes
    for mes in resumo:
        vals = resumo[mes]['valores']
        resumo[mes]['saldo']         = round(resumo[mes]['total_credito'] - resumo[mes]['total_debito'], 2)
        resumo[mes]['media']         = round(sum(vals) / len(vals), 2)
        resumo[mes]['maior_valor']   = max(vals)
        resumo[mes]['menor_valor']   = min(vals)
        resumo[mes]['total_credito'] = round(resumo[mes]['total_credito'], 2)
        resumo[mes]['total_debito']  = round(resumo[mes]['total_debito'], 2)
        del resumo[mes]['valores']

    return dict(sorted(resumo.items()))


def salvar_json(relatorio, n_validas, n_invalidas):
    # converte datetime pra string antes de salvar
    suspeitas_json = []
    for t in relatorio['suspeitas']:
        suspeitas_json.append({
            'id': t['id'],
            'cliente_id': t['cliente_id'],
            'data': t['data'].strftime('%Y-%m-%d'),
            'valor': t['valor'],
            'descricao': t['descricao']
        })

    payload = {
        'gerado_em': datetime.now().strftime('%Y-%m-%d'),
        'total_transacoes_validas': n_validas,
        'total_transacoes_invalidas': n_invalidas,
        'resumo_mensal': relatorio['resumo'],
        'transacoes_suspeitas': suspeitas_json
    }

    with open('relatorio.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print("relatorio.json salvo!")


def exibir_relatorio(relatorio, n_validas, n_invalidas, periodo, dias):
    print("\n===== RESUMO DA LEITURA =====")
    print(f"Periodo analisado: {periodo[0]} ate {periodo[1]} ({dias} dias)")
    print(f"Total de linhas lidas: {n_validas + n_invalidas}")
    print(f"Linhas validas: {n_validas}")
    print(f"Linhas invalidas: {n_invalidas}")

    print("\n===== RELATORIO MENSAL =====")
    for mes, dados in relatorio['resumo'].items():
        print(f"\nMes: {mes}")
        print(f"  Transacoes: {dados['quantidade']}")
        print(f"  Total credito: {brl(dados['total_credito'])}")
        print(f"  Total debito:  {brl(dados['total_debito'])}")
        print(f"  Saldo:         {brl(dados['saldo'])}")
        print(f"  Media:         {brl(dados['media'])}")
        print(f"  Maior valor:   {brl(dados['maior_valor'])}")
        print(f"  Menor valor:   {brl(dados['menor_valor'])}")

    print("\n===== TRANSACOES SUSPEITAS =====")
    if relatorio['suspeitas']:
        for t in relatorio['suspeitas']:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {t['data'].strftime('%Y-%m-%d')} | Valor: {brl(t['valor'])}")
    else:
        print("Nenhuma transacao suspeita encontrada.")


print("funcoes definidas!")


funcoes definidas!


## Executando a análise

In [4]:
# lendo o arquivo
linhas = ler_transacoes('transacoes.csv')

# validando cada linha
validas = []
invalidas = 0

for linha in linhas:
    resultado = validar_transacao(linha)
    if resultado:
        validas.append(resultado)
    else:
        invalidas += 1

print(f"Total de linhas lidas: {len(linhas)}")
print(f"Linhas validas: {len(validas)}")
print(f"Linhas invalidas: {invalidas}")

# separando as suspeitas
suspeitas = [t for t in validas if t['valor'] > LIMITE_SUSPEITO]

# calculando periodo e quantidade de dias
datas = [t['data'] for t in validas]
data_min = min(datas)
data_max = max(datas)
dias = (data_max - data_min).days
periodo = (data_min.strftime('%Y-%m-%d'), data_max.strftime('%Y-%m-%d'))

# gerando o resumo mensal
resumo = gerar_relatorio(validas)

relatorio = {
    'resumo': resumo,
    'suspeitas': suspeitas
}

# salvando e exibindo
salvar_json(relatorio, len(validas), invalidas)
exibir_relatorio(relatorio, len(validas), invalidas, periodo, dias)


Total de linhas lidas: 21
Linhas validas: 16
Linhas invalidas: 5
relatorio.json salvo!

===== RESUMO DA LEITURA =====
Periodo analisado: 2026-01-05 ate 2026-05-15 (130 dias)
Total de linhas lidas: 21
Linhas validas: 16
Linhas invalidas: 5

===== RELATORIO MENSAL =====

Mes: 2026-01
  Transacoes: 2
  Total credito: R$ 3.500,00
  Total debito:  R$ 180,50
  Saldo:         R$ 3.319,50
  Media:         R$ 1.840,25
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 180,50

Mes: 2026-02
  Transacoes: 2
  Total credito: R$ 15.000,00
  Total debito:  R$ 320,00
  Saldo:         R$ 14.680,00
  Media:         R$ 7.660,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 320,00

Mes: 2026-03
  Transacoes: 4
  Total credito: R$ 6.300,00
  Total debito:  R$ 549,90
  Saldo:         R$ 5.750,10
  Media:         R$ 1.712,47
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90

Mes: 2026-04
  Transacoes: 5
  Total credito: R$ 17.200,00
  Total debito:  R$ 965,00
  Saldo:         R$ 16.235,00
  Media:    